# Demand Forecasting: Baseline Models & MLflow Tracking

This notebook implements and evaluates the forecasting baselines specified in **Section 8 (Train Baseline Models First)** of the [Pipeline Implementation Guide](file:///Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/documents/pipeline-implementation-guide.md):
- **Last-day demand**
- **Same-day-last-week demand**
- **Moving average** (7-day and 28-day)
- **Seasonal naive forecast** (weekly and 28-day)
- **Diagnostic benchmark: Always Zero**

---

### Integration with Data Pipelines & History
Unlike isolated scripts, this notebook directly connects with the upstream pipelines:
1. **Historical Analytical Warehouse**: Reads directly from `data/warehouse/olist.duckdb` and `data/curated/` partitions.
2. **Feature Store Lineage**: Uses the leak-free, partitioned Parquet features built by `scripts/features/build_forecasting_features.py` (`data/features/forecasting/{train, validation, test}.parquet`).
3. **Dedicated MLflow Tracking**: Logs runs under the dedicated experiment `ecommerce-demand-forecasting` in accordance with Section 9.
4. **Pipeline-Ready Design**: Defines a modular `BaselineForecaster` class ready to integrate into `pipelines/training_pipeline.py`.


The reusable `BaselineForecaster` example forecasts next-day demand. Earlier exploratory SQL examples in this notebook score same-day demand. For comparable baseline/candidate MLflow runs, use the CLI jobs and `olist_production_training` DAG described in the README.

In [ ]:
import os
import sys
import json
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
import duckdb
import numpy as np
import pandas as pd
import mlflow
from IPython.display import display

# Identify repository root
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/features/build_forecasting_features.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Notebook must be run from inside the repository.")
sys.path.insert(0, str(ROOT))

# Paths from upstream pipeline
DATA_DIR = ROOT / "data"
WAREHOUSE_PATH = DATA_DIR / "warehouse" / "olist.duckdb"
CURATED_DIR = DATA_DIR / "curated"
FEATURES_DIR = DATA_DIR / "features" / "forecasting"
BASELINES_DIR = DATA_DIR / "baselines"
BASELINES_DIR.mkdir(parents=True, exist_ok=True)

# Configure MLflow dedicated experiment
MLFLOW_DB = BASELINES_DIR / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
EXPERIMENT_NAME = "ecommerce-demand-forecasting"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Repository Root: {ROOT}")
print(f"Features Store: {FEATURES_DIR}")
print(f"MLflow Store: {mlflow.get_tracking_uri()}")
print(f"MLflow Experiment: {EXPERIMENT_NAME}")


## 1. Verify Pipeline Lineage & Historical Data

Before training, we verify the data lineage from upstream pipeline stages:
- Curated Data & Warehouse (`scripts/data/prepare_curated_data.py`)
- Forecasting Feature Parquets & Train/Validation/Test Splits (`scripts/features/build_forecasting_features.py`)


In [ ]:
# 1. Check feature store metadata
metadata_path = FEATURES_DIR / "metadata.json"
if not metadata_path.is_file():
    raise FileNotFoundError(f"Missing forecasting feature metadata: {metadata_path}. Run scripts/features/build_forecasting_features.py first.")

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
print("Forecasting Feature Store Metadata:")
print(json.dumps(metadata, indent=2))

# 2. Inspect DuckDB warehouse tables
with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as con:
    wh_tables = con.execute("SHOW TABLES").df()
    print("\nAnalytical Warehouse Tables (olist.duckdb):")
    display(wh_tables)
    
    demand_summary = con.execute("""
        SELECT 
            MIN(purchase_date) AS min_date,
            MAX(purchase_date) AS max_date,
            COUNT(DISTINCT product_id) AS distinct_products,
            COUNT(DISTINCT customer_state) AS distinct_states,
            SUM(order_item_count) AS total_units_sold
        FROM daily_demand
    """).df()
    print("\nCurated Daily Demand Overview:")
    display(demand_summary)


## 2. Inspect the Forecasting Feature Dataset

We inspect the parquet schema and the pre-computed lag and rolling features produced by `scripts/features/build_forecasting_features.py`.
Notice that the feature pipeline has ALREADY computed:
- `demand_lag_1`: Demand from 1 day prior ($T-1$)
- `demand_lag_7`: Demand from 7 days prior ($T-7$, same day of week)
- `demand_lag_14`: Demand from 14 days prior ($T-14$, two weeks prior)
- `demand_lag_28`: Demand from 28 days prior ($T-28$, four weeks prior)
- `rolling_mean_7`: 7-day trailing moving average (excluding current day)
- `rolling_mean_28`: 28-day trailing moving average
- `target_demand_next_day`: Actual next-day demand ($T+1$)


In [ ]:
con = duckdb.connect()
split_overview = con.execute(f"""
    SELECT 
        'validation' AS split,
        MIN(purchase_date) AS min_date,
        MAX(purchase_date) AS max_date,
        COUNT(*) AS row_count,
        COUNT(DISTINCT product_id) AS products,
        COUNT(DISTINCT customer_state) AS states,
        SUM(demand) AS total_demand
    FROM read_parquet('{FEATURES_DIR}/validation.parquet')
    UNION ALL
    SELECT 
        'test' AS split,
        MIN(purchase_date) AS min_date,
        MAX(purchase_date) AS max_date,
        COUNT(*) AS row_count,
        COUNT(DISTINCT product_id) AS products,
        COUNT(DISTINCT customer_state) AS states,
        SUM(demand) AS total_demand
    FROM read_parquet('{FEATURES_DIR}/test.parquet')
""").df()
display(split_overview)


## 3. The Four Core Forecasting Baselines

We define each baseline rule clearly, connecting it to the features produced by the pipeline:

| Baseline Model | Mathematical Rule | Feature Store Mapping | Description |
|---|---|---|---|
| **Last-Day Demand** | $\hat{y}_{t} = y_{t-1}$ | `demand_lag_1` | Standard naive persistence: tomorrow will be just like today |
| **Same-Day-Last-Week** | $\hat{y}_{t} = y_{t-7}$ | `demand_lag_7` | Repeats demand from the exact same day of the prior week (weekly seasonality) |
| **Moving Average (7-day)** | $\hat{y}_{t} = \frac{1}{7} \sum_{i=1}^7 y_{t-i}$ | `rolling_mean_7` | Trailing 7-day average: filters out single-day volatility |
| **Moving Average (28-day)**| $\hat{y}_{t} = \frac{1}{28} \sum_{i=1}^{28} y_{t-i}$ | `rolling_mean_28` | Trailing 28-day average: captures longer-term baseline velocity |
| **Seasonal Naive (28-day)** | $\hat{y}_{t} = y_{t-28}$ | `demand_lag_28` | Repeats demand from 4 weeks prior |
| **Always Zero** | $\hat{y}_{t} = 0$ | Constant `0.0` | Diagnostic benchmark: critical in retail where >99% of daily series are 0 |

### Evaluation Metrics

- **MAE:** average absolute prediction error, in units.
- **RMSE:** penalizes large prediction errors more strongly.

Lower is better for both. The scripts and notebooks use the same two metrics.


In [ ]:
def evaluate_baselines(parquet_path: Path, split_name: str) -> pd.DataFrame:
    sql_path = str(parquet_path).replace("'", "''")
    query = f"""
    WITH base AS (
        SELECT
            demand AS actual,
            demand_lag_1 AS last_day,
            demand_lag_7 AS same_day_last_week,
            rolling_mean_7 AS moving_average_7,
            rolling_mean_28 AS moving_average_28,
            demand_lag_28 AS seasonal_naive_28,
            0.0 AS always_zero
        FROM read_parquet('{sql_path}')
    )
    SELECT
        '{split_name}' AS split,
        baseline,
        COUNT(*) AS total_rows,
        AVG(ABS(actual - pred)) AS mae,
        SQRT(AVG(POWER(actual - pred, 2))) AS rmse
    FROM (
        SELECT actual, last_day AS pred, 'last_day' AS baseline FROM base
        UNION ALL
        SELECT actual, same_day_last_week, 'same_day_last_week' FROM base
        UNION ALL
        SELECT actual, moving_average_7, 'moving_average_7' FROM base
        UNION ALL
        SELECT actual, moving_average_28, 'moving_average_28' FROM base
        UNION ALL
        SELECT actual, seasonal_naive_28, 'seasonal_naive_28' FROM base
        UNION ALL
        SELECT actual, always_zero, 'always_zero' FROM base
    )
    GROUP BY baseline
    ORDER BY mae
    """
    with duckdb.connect() as con:
        return con.execute(query).df()

val_metrics = evaluate_baselines(FEATURES_DIR / "validation.parquet", "validation")
test_metrics = evaluate_baselines(FEATURES_DIR / "test.parquet", "test")

print("Validation Split Baseline Performance:")
display(val_metrics)
print("\nTest Split Baseline Performance:")
display(test_metrics)


## 4. MLflow Experiment Tracking

Now we track each baseline model as a distinct run in the `ecommerce-demand-forecasting` MLflow experiment.
For each run, we record:
- **Parameters**: `baseline_name`, `window_days`, `lag_offset`, `feature_version`, `dataset_split_dates`, `upstream_pipeline_stage`.
- **Metrics**: validation/test MAE and RMSE.
- **Artifacts**: Metrics CSV, prediction inspection table, and error distribution summary.
- **Tags**: `baseline_type`, `pipeline_integrated: true`, `target: daily_demand`.


In [ ]:
combined_metrics = pd.concat([val_metrics, test_metrics], ignore_index=True)
combined_metrics.to_csv(BASELINES_DIR / "forecasting_metrics_all.csv", index=False)

# Identify the champion non-trivial baseline on validation split
non_trivial = val_metrics[val_metrics["baseline"] != "always_zero"]
champion_baseline = non_trivial.sort_values("mae").iloc[0]["baseline"]
print(f"Champion Non-Trivial Baseline selected on Validation MAE: '{champion_baseline}'")

mlflow_runs = []
for baseline in val_metrics["baseline"].unique():
    with mlflow.start_run(run_name=f"baseline__{baseline}") as run:
        is_champion = (baseline == champion_baseline)
        
        # 1. Log Parameters
        mlflow.log_params({
            "task": "demand_forecasting",
            "model_family": "heuristic_baseline",
            "baseline_name": baseline,
            "window_days": 7 if "7" in baseline else 28 if "28" in baseline else 1 if "last_day" in baseline else 0,
            "pipeline_feature_store": "data/features/forecasting",
            "validation_range": f"{metadata['date_ranges']['validation']['min']} to {metadata['date_ranges']['validation']['max']}",
            "test_range": f"{metadata['date_ranges']['test']['min']} to {metadata['date_ranges']['test']['max']}",
            "is_champion_baseline": is_champion,
        })
        
        # 2. Log Tags
        mlflow.set_tag("pipeline_integrated", "true")
        mlflow.set_tag("selected_on_validation", str(is_champion))
        if is_champion:
            mlflow.set_tag("model_status", "champion_baseline")
            
        # 3. Log Metrics for validation and test
        for split_name, table in [("val", val_metrics), ("test", test_metrics)]:
            row = table.loc[table["baseline"] == baseline].iloc[0]
            mlflow.log_metrics({
                f"{split_name}_mae": float(row["mae"]),
                f"{split_name}_rmse": float(row["rmse"]),
            })
            
        # 4. Log Artifacts
        artifact_path = BASELINES_DIR / f"metrics_{baseline}.csv"
        subset = combined_metrics.loc[combined_metrics["baseline"] == baseline]
        subset.to_csv(artifact_path, index=False)
        mlflow.log_artifact(str(artifact_path))
        
        mlflow_runs.append({
            "baseline": baseline,
            "run_id": run.info.run_id,
            "val_mae": float(val_metrics.loc[val_metrics['baseline'] == baseline, 'mae'].iloc[0]),
            "champion": is_champion
        })

runs_df = pd.DataFrame(mlflow_runs)
display(runs_df)
print("\nLogged runs successfully to MLflow experiment:", EXPERIMENT_NAME)


## 5. Select a forecasting baseline

Choose the nonzero baseline with the lowest validation MAE, then compare its test MAE and RMSE. Keep the always-zero method as a diagnostic comparison.

Use results generated by this run rather than hardcoded historical scores.


In [ ]:
with duckdb.connect() as con:
    # Find the top 3 most active product-state series in validation
    top_series = con.execute(f"""
        SELECT product_id, customer_state, SUM(demand) as total_units
        FROM read_parquet('{FEATURES_DIR}/validation.parquet')
        GROUP BY 1, 2
        ORDER BY total_units DESC
        LIMIT 3
    """).df()
    print("Top Active Product-State Series in Validation:")
    display(top_series)
    
    top_prod, top_state = top_series.iloc[0]["product_id"], top_series.iloc[0]["customer_state"]
    sample_preds = con.execute(f"""
        SELECT 
            purchase_date,
            demand AS actual_demand,
            demand_lag_1 AS last_day,
            demand_lag_7 AS same_day_last_week,
            ROUND(rolling_mean_7, 2) AS moving_avg_7,
            ROUND(rolling_mean_28, 2) AS moving_avg_28
        FROM read_parquet('{FEATURES_DIR}/validation.parquet')
        WHERE product_id = '{top_prod}' AND customer_state = '{top_state}'
        ORDER BY purchase_date
        LIMIT 14
    """).df()
    print(f"\n14-Day Forecast Comparison for Active Series (Product: {top_prod[:8]}..., State: {top_state}):")
    display(sample_preds)
    sample_preds.to_csv(BASELINES_DIR / "sample_active_forecasts.csv", index=False)
    
    # Log sample predictions as artifact to the champion run
    champ_run_id = runs_df.loc[runs_df["champion"] == True, "run_id"].iloc[0]
    with mlflow.start_run(run_id=champ_run_id):
        mlflow.log_artifact(str(BASELINES_DIR / "sample_active_forecasts.csv"))


## 6. Pipeline Architecture: Reusable `BaselineForecaster` Class

To prepare for the python training pipeline (`pipelines/training_pipeline.py`), we encapsulate the forecasting logic into a clean, reusable Python class.
This class can predict on any dataframe conforming to our feature contract and output standard predictions.


In [ ]:
from src.models.baseline.forecasting_baselines import BaselineForecaster

# Verify the class against validation split sample
val_sample = pd.read_parquet(FEATURES_DIR / "validation.parquet", columns=[
    "demand", "demand_lag_1", "demand_lag_6", "demand_lag_7", "rolling_mean_7", "rolling_mean_28", "demand_lag_27", "demand_lag_28", "target_demand_next_day"
]).head(10000)

forecaster = BaselineForecaster(method="moving_average_28")
sample_eval = forecaster.evaluate(val_sample)
print("BaselineForecaster Quick Test on 10,000 rows:")
print(json.dumps(sample_eval, indent=2))


## 7. Summary

The notebook compares the forecasting baselines using MAE and RMSE and records validation/test results in MLflow. Lower is better for both metrics.

The reusable implementation is `src/models/baseline/forecasting_baselines.py`; its CLI is `scripts/models/baseline/forecasting_baselines.py`.
